## Biomass LA2 product Congo basin vs CCI Biomass
Objetive: Validate different areas of the Congo basin using the Biomass LA2 map against the information of the CCI biomass product

### 1. Packages and Libraries

In [2]:
%pip install aiohttp requests

from shapely.geometry import Polygon, box
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

from rasterio.windows import from_bounds
from rasterio.plot import show
from rasterio.plot import plotting_extent
from shapely.geometry import shape
from rasterio.enums import Resampling
import rioxarray  # adds .rio to xarray

  Using cached aiohttp-3.14.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-7.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (6.2 kB)
  Using cached propcache-0.5.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (25 kB)
  Using cached yarl-1.25.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (111 kB)
Using cached aiohttp-3.14.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.9 MB)
Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.

## 2. Open AGB LA2 Congo map

In [3]:
# Path to your GeoTIFF file
tif_path = r'Examples_and_Data/Workshops/BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif'

with rio.open(tif_path) as src:
    print("File:", src.name)
    print("Raster dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate reference system:", src.crs)
    print("Pixel resolution:", src.res)
    print("No-data value:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Bounds:", src.bounds)

agb_bbox = [coord for coord in src.bounds]
agb_bbox

File: Examples_and_Data/Workshops/BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif
Raster dimensions: 27103 x 16237
Number of bands: 7
Coordinate reference system: EPSG:4326
Pixel resolution: (0.0017980138121629352, 0.0018087108088100656)
No-data value: nan
Band descriptions: ('AGBD estimate [t/ha]', '10log10(AGBD) estimate [dB]', '10log10(AGBD) standard deviation [dB]', 'AGBD lower one-sigma [t/ha]', 'AGBD upper one-sigma [t/ha]', 'valid acquisition count', 'valid band-observation count')
Bounds: BoundingBox(left=4.87621345858588, bottom=-12.429460678142771, right=53.60778180963791, top=16.938576724506266)


[4.87621345858588, -12.429460678142771, 53.60778180963791, 16.938576724506266]

## 3. Query the ESA MAAP STAC catalog

In [4]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

### 3.1.  Select a collection and narrow down the search
Using the bounding box of the AGB map from Congo get the CCI Biomass items

In [5]:
search = catalog.search(
    collections = ["CCIBiomassV5.01"],
    datetime = ['2021-01-01T00:00:00Z','2021-12-31T00:00:00Z'],
    bbox = agb_bbox,
    method = "GET",
    max_items = 50,  
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Accessing 44 items (limited by max_items).
44 items found that matched the query.


Generate the gdf with the information of CCI biomass available

In [6]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    if "L4-AGB-MERGED" in it.id and "SD_MERGED" not in it.id:
        props = it.properties or {}
        features.append({
            "type": "Feature",
            "geometry": it.geometry,
            "properties": {
                "product_id": it.id,
                "start_datetime": props.get("start_datetime"),
                "end_datetime": props.get("end_datetime"),
                # "item": it
            },
        })


gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
# gdf = gdf[["index", "product_id", "start_datetime", "end_datetime", "geometry"]]
gdf = gdf[["product_id", "start_datetime", "end_datetime", "geometry"]]
gdf.head(len(items))

,product_id,start_datetime,end_datetime,geometry
0,N00E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((30 -10, 40 -10, 40 0, 30 0, 30 -10))"
1,N00E040_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((40 -10, 50 -10, 50 0, 40 0, 40 -10))"
2,N00E020_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((20 -10, 30 -10, 30 0, 20 0, 20 -10))"
3,N00E010_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((10 -10, 20 -10, 20 0, 10 0, 10 -10))"
4,N10E020_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((20 0, 30 0, 30 10, 20 10, 20 0))"
5,N20E000_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((0 10, 10 10, 10 20, 0 20, 0 10))"
6,N20E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((30 10, 40 10, 40 20, 30 20, 30 10))"
7,N20E040_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((40 10, 50 10, 50 20, 40 20, 40 10))"
8,N20E020_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((20 10, 30 10, 30 20, 20 20, 20 10))"
9,S10E020_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021...,2021-01-01T00:00:00.000Z,2021-12-31T23:59:59.000Z,"POLYGON ((20 -20, 30 -20, 30 -10, 20 -10, 20 -..."


In [7]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

**You need to create a .txt file and just paste your token inside**

In [8]:
if pathlib.Path("GB_access_token.txt").exists(): # Put here your .txt file path
  with open("GB_access_token.txt","rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Access token retrieved successfully.


## 4. Apply the pipeline to each CCI product

In [9]:
def crop_tif(path_tif, prod_id, bounds, path_folder_save):
    minx, miny, maxx, maxy = bounds
    # Replace these example coordinates with your desired zone
    left = minx
    bottom = miny
    right = maxx
    top = maxy
    with rio.open(path_tif) as src:
        # Convert map coordinates into a pixel window
        window = from_bounds(
            left, bottom, right, top,
            transform=src.transform
        )
        # Align the window to whole pixels
        window = window.round_offsets().round_lengths()
        # Crop the raster
        cropped = src.read(1, window=window, masked=True)
        # Update the spatial transform for the cropped raster
        cropped_transform = src.window_transform(window)
        print("Original size:", src.width, "x", src.height)
        print("Cropped size:", cropped.shape[1], "x", cropped.shape[0])
        # Calculate the correct map extent for the cropped zone
        extent = plotting_extent(cropped, cropped_transform)
        # Convert masked pixels to NaN for display
        cropped_data = cropped.filled(np.nan)
        valid = cropped_data[np.isfinite(cropped_data)]
        valid_pxs = valid.size
        if valid_pxs > 0:
            out_meta = src.meta
            cropped_data_nx = np.expand_dims(cropped_data, axis=0)
            cropped_data_nx.shape
            out_meta.update({"driver": "GTiff",
                 "height": cropped_data_nx.shape[1],
                 "width": cropped_data_nx.shape[2],
                 "transform": cropped_transform,
                 "count":1})
            path_tif_save = os.path.join(path_folder_save,f"agbd_map_L2A_cropped_{prod_id}.tif")
            with rio.open(os.path.join(path_folder_save,f"agbd_map_L2A_cropped_{prod_id}.tif"), "w", **out_meta) as dest:
                dest.write(cropped_data_nx)
            print(f'Tiff saved: {path_tif_save}')
            return path_tif_save
        else:
            print(f'Not valid pixels N: {valid_pxs}')
            return 0          

In [10]:
path_folder_save = r'temp_tif'

In [11]:
df_stats = pd.DataFrame()
for i, row in gdf.iterrows():
    prod_id = row['product_id']
    id_coords = prod_id.split('_')[0]
    cci_i_bounds = row['geometry'].bounds
    print(i, prod_id, cci_i_bounds)
    biomass_product = [id_ for id_ in items if prod_id in str(id_)][0]
    # Crop the Biomass LA2 map 
    path_tif_cropped = crop_tif(tif_path, prod_id, cci_i_bounds, path_folder_save)
    if path_tif_cropped == 0:
        continue
    else:
       # Open the agb map cropped with rioxarray
        agb_cropped = rioxarray.open_rasterio(path_tif_cropped, masked=True).squeeze().load()
        # cut the BIOMASS frame to the ALS tile
        agb_bounds = agb_cropped.rio.transform_bounds("EPSG:4326") 
        # Create an authenticated HTTPS filesystem for streaming access
        fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {access_token}"})
        
        # Open the remote raster file as a binary stream
        with fs.open(biomass_product.assets['enclosure_tif'].href, "rb") as f:
            cci_agb = (rioxarray.open_rasterio(f, masked=True).squeeze().rio.clip_box(*agb_bounds, crs="EPSG:4326").load())
        for name, da in [("Biomass LA2", agb_cropped), ("CCI biomass", cci_agb)]:
            print(f"{name:8s} CRS={da.rio.crs}  res={da.rio.resolution()}  shape={da.shape}  "
                  f"range=({float(da.min()):.1f}, {float(da.max()):.1f})")
        cci_on_agb_mean = cci_agb.rio.reproject_match(agb_cropped, resampling=Resampling.average)
        cci_on_agb_max = cci_agb.rio.reproject_match(agb_cropped, resampling=Resampling.max)
        valid = agb_cropped.notnull().astype("float32").rio.write_nodata(np.nan)
        coverage = valid.rio.reproject_match(agb_cropped, resampling=Resampling.average)
        
        MIN_COVER = 0.8
        cmp = pd.DataFrame({
            f"{id_coords}_CCI_biomass_mean": cci_on_agb_mean.values.ravel(),
            f"{id_coords}_CCI_biomass_max": cci_on_agb_max.values.ravel(),
            f"{id_coords}_Biomass_LA2": agb_cropped.values.ravel(),
            f"{id_coords}_cover": coverage.values.ravel(),
        }).dropna()
        cmp = cmp[cmp[f"{id_coords}_cover"] >= MIN_COVER]
        print(f"Matched BIOMASS pixels with >={MIN_COVER:.0%} CCI biomass coverage: {len(cmp)}")
        
        def stats(y_ref, y_est):
            d = y_est - y_ref
            r = np.corrcoef(y_ref, y_est)[0, 1]
            return {"n": len(d), "bias": d.mean(), "RMSE": np.sqrt((d**2).mean()),
                    "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}
        
        
        df_stats_ = pd.DataFrame({k: stats(cmp[k], cmp[f"{id_coords}_Biomass_LA2"]) for k in [f"{id_coords}_CCI_biomass_mean",f"{id_coords}_CCI_biomass_max"]}).T.round(2)
        df_stats = pd.concat([df_stats,df_stats_], axis=0)
        # Plots
        # if df_stats_.loc[0,'R2'] > 0.5:
        #     fig, ax = plt.subplots(1, 3, figsize=(16, 5))
        #     vmax = float(np.nanpercentile(agb_cropped, 99))
        #     cci_on_agb_mean.plot(ax=ax[0], vmin=0, vmax=vmax, cmap="viridis"); ax[0].set_title(f"CCI biomass {id_coords}")
        #     agb_cropped.plot(ax=ax[1], vmin=0, vmax=vmax, cmap="viridis"); ax[1].set_title("BIOMASS L2A (mean, on CCI biomass grid)")
        #     (cci_on_agb_mean - agb_cropped).plot(ax=ax[2], cmap="RdBu_r", center=0); ax[2].set_title("BIOMASS − ALS")
        #     for a in ax:
        #         a.set_aspect("equal")
        #     plt.tight_layout()
        #     plt.show()
    # break

0 N00E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (30.0, -10.0, 40.0, 0.0)
Original size: 27103 x 16237
Cropped size: 5562 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N00E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 5562)  range=(0.1, 1800.0)
CCI biomass CRS=EPSG:4326  res=(0.0008888888888888894, -0.000888888888888889)  shape=(11250, 11250)  range=(0.0, 370.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 12176036
1 N00E040_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (40.0, -10.0, 50.0, 0.0)
Original size: 27103 x 16237
Cropped size: 5562 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N00E040_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 5562)  range=(0.1, 1592.0)
CCI biomass CRS=EPSG:4326  res=(0.000888888888888889, -0.000888888888888889)  shape=(11250, 11250)  range=(

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_113/3802519178.py:45: RuntimeWarning: divide by zero encountered in scalar divide
  "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}


Original size: 27103 x 16237
Cropped size: 2007 x 1344
Not valid pixels N: 0
15 N00E000_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (0.0, -10.0, 10.0, 0.0)
Original size: 27103 x 16237
Cropped size: 2850 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N00E000_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 2850)  range=(0.1, 1800.0)
CCI biomass CRS=EPSG:4326  res=(0.0008888888888888889, -0.000888888888888889)  shape=(11250, 5765)  range=(0.0, 0.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 2994900
16 N10E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (30.0, 0.0, 40.0, 10.0)


/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_113/3802519178.py:45: RuntimeWarning: divide by zero encountered in scalar divide
  "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}


Original size: 27103 x 16237
Cropped size: 5562 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N10E030_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 5562)  range=(0.1, 1800.0)
CCI biomass CRS=EPSG:4326  res=(0.0008888888888888894, -0.000888888888888889)  shape=(11250, 11250)  range=(0.0, 532.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 23842468
17 N10E050_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (50.0, 0.0, 60.0, 10.0)
Original size: 27103 x 16237
Cropped size: 2007 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N10E050_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 2007)  range=(1.3, 525.7)
CCI biomass CRS=EPSG:4326  res=(0.0008888888888888885, -0.000888888888888889)  shape=(11250, 4059)  range=(0.0, 64.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 172869
18 N10E

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_113/3802519178.py:45: RuntimeWarning: divide by zero encountered in scalar divide
  "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}


Original size: 27103 x 16237
Cropped size: 5562 x 5529
Tiff saved: temp_tif/agbd_map_L2A_cropped_N10E010_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(5529, 5562)  range=(0.1, 1800.0)
CCI biomass CRS=EPSG:4326  res=(0.000888888888888889, -0.000888888888888889)  shape=(11250, 11250)  range=(0.0, 832.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 29235180
19 N20E010_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0 (10.0, 10.0, 20.0, 20.0)
Original size: 27103 x 16237
Cropped size: 5562 x 3836
Tiff saved: temp_tif/agbd_map_L2A_cropped_N20E010_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2021-fv5.0.tif
Biomass LA2 CRS=EPSG:4326  res=(0.0017980138121629352, -0.0018087108088100656)  shape=(3836, 5562)  range=(0.1, 1800.0)
CCI biomass CRS=EPSG:4326  res=(0.000888888888888889, -0.0008888888888888889)  shape=(7805, 11250)  range=(0.0, 64.0)
Matched BIOMASS pixels with >=80% CCI biomass coverage: 12848772
20 S

In [12]:
# Select the elements with enough pixels to validate
df = df_stats[(~df_stats['R2'].isna()) & (df_stats['R2'] > 0.01)]
df

,n,bias,RMSE,rel_RMSE_%,R2
N00E030_CCI_biomass_mean,12176036.0,25.13,70.90,508.47,0.46
N00E030_CCI_biomass_max,12176036.0,16.63,65.23,290.60,0.45
N00E040_CCI_biomass_mean,3115394.0,6.54,22.23,1692.67,0.19
N00E040_CCI_biomass_max,3115394.0,5.78,21.50,1038.93,0.21
N00E020_CCI_biomass_mean,23950461.0,4.34,107.97,64.18,0.57
N00E020_CCI_biomass_max,23950461.0,-49.89,132.22,59.44,0.57
N00E010_CCI_biomass_mean,19657933.0,-7.04,83.33,51.46,0.73
N00E010_CCI_biomass_max,19657933.0,-60.44,117.97,54.79,0.71
N10E020_CCI_biomass_mean,29810907.0,-16.47,93.31,54.04,0.64
N10E020_CCI_biomass_max,29810907.0,-71.15,140.15,61.65,0.62


In [17]:
# Save the df of the validation
path_save_csv = r'Validation_stats.csv'
df.to_csv(path_save_csv)

In [16]:
# Select the mean idx for the mean biomass la2 values
idxs_mean = [idx_ for idx_ in df.index.values if 'mean' in idx_]

df_means = df.loc[idxs_mean]

df_means

,n,bias,RMSE,rel_RMSE_%,R2
N00E030_CCI_biomass_mean,12176036.0,25.13,70.90,508.47,0.46
N00E040_CCI_biomass_mean,3115394.0,6.54,22.23,1692.67,0.19
N00E020_CCI_biomass_mean,23950461.0,4.34,107.97,64.18,0.57
N00E010_CCI_biomass_mean,19657933.0,-7.04,83.33,51.46,0.73
N10E020_CCI_biomass_mean,29810907.0,-16.47,93.31,54.04,0.64
S10E020_CCI_biomass_mean,4766356.0,39.28,74.34,168.32,0.57
S10E010_CCI_biomass_mean,1893419.0,64.37,98.38,177.73,0.62
N10E040_CCI_biomass_mean,12556138.0,15.57,51.00,1089.90,0.37
N10E030_CCI_biomass_mean,23842468.0,29.36,109.17,491.52,0.22
N10E010_CCI_biomass_mean,29235180.0,-7.01,102.31,56.07,0.60


In [37]:
df_means['R2'].mean()

0.5054545454545454